# ECDT — Phase 2 : Ingestion, Normalisation et Détection d'Anomalies

## Objectif

Ce notebook valide l'étape 2 de l'architecture ECDT.

La chaîne testée est :

```text
RCAEval / données préparées
        |
        v
DatasetLoader
        |
        v
SchemaNormalizer
        |
        v
Normalized Events
        |
        v
AnomalyDetector
        |
        v
Time Series + Baseline
        |
        v
Anomaly Events
        |
        v
Validation Ground Truth
```

L'objectif n'est pas encore de réaliser la RCA complète. Il s'agit de vérifier que les données peuvent être chargées, normalisées, représentées sous forme de séries temporelles et utilisées pour détecter les anomalies autour de l'injection connue.

Les données de Phase 1 utilisées ici correspondent au sous-ensemble ECDT de 60 incidents : 15 CPU, 15 DELAY, 15 LOSS et 15 SOCKET. :contentReference[oaicite:2]{index=2}

## 1. Préparation de l'environnement

Le notebook doit être exécuté depuis la racine du projet ECDT ou avec la racine correctement détectée.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("src exists:", (PROJECT_ROOT / "src").exists())

Project root: C:\Users\ou2_s\Documents\GitHub\ECDT
Python: c:\Users\ou2_s\Documents\GitHub\ECDT\.venv\Scripts\python.exe
src exists: True


## 2. Imports ECDT


In [2]:
import polars as pl

from src.ingestion.dataset_loader import create_default_loader
from src.ingestion.schema_normalizer import SchemaNormalizer
from src.ingestion.anomaly_detector import AnomalyDetector

print("Polars:", pl.__version__)
print("DatasetLoader: OK")
print("SchemaNormalizer: OK")
print("AnomalyDetector: OK")

Polars: 1.43.2
DatasetLoader: OK
SchemaNormalizer: OK
AnomalyDetector: OK


## 3. Initialisation


In [3]:
loader = create_default_loader(PROJECT_ROOT)
normalizer = SchemaNormalizer()
detector = AnomalyDetector()

print("ECDT Phase 2 components initialized successfully.")

ECDT Phase 2 components initialized successfully.


## 4. Vérification du ground truth

Le fichier `target_incidents.csv` constitue la référence temporelle et causale utilisée pour valider les détections.


In [4]:
cases = loader.list_cases()

print("Nombre de cas disponibles:", len(cases))
print("Premiers cas:")
for case in cases[:10]:
    print(" -", case)

Nombre de cas disponibles: 60
Premiers cas:
 - re2ob_checkoutservice_cpu_1
 - re2ob_checkoutservice_delay_1
 - re2ob_checkoutservice_loss_1
 - re2ob_checkoutservice_socket_1
 - re2ob_currencyservice_cpu_1
 - re2ob_currencyservice_delay_1
 - re2ob_currencyservice_loss_1
 - re2ob_currencyservice_socket_1
 - re2ob_emailservice_cpu_1
 - re2ob_emailservice_delay_1


## 5. Répartition des incidents


In [5]:
ground_truth = loader.load_ground_truth()

print("Shape:", ground_truth.shape)
print("Colonnes:")
print(ground_truth.columns)

display(ground_truth.head())

Shape: (60, 13)
Colonnes:
['case', 'dataset', 'suite', 'system_name', 'fault', 'fault_description', 'root_cause_service', 'inject_time', 'time_start', 'time_end', 'duration_minutes', 'normal_timesteps', 'faulty_timesteps']


case,dataset,suite,system_name,fault,fault_description,root_cause_service,inject_time,time_start,time_end,duration_minutes,normal_timesteps,faulty_timesteps
str,str,str,str,str,str,str,i64,i64,i64,f64,i64,i64
"""re2ob_checkoutservice_cpu_1""","""RE2-OB""","""RE2""","""Online Boutique""","""cpu""","""CPU stress""","""checkoutservice""",1705354566,1705353846,1705355286,24.0,720,721
"""re2ob_checkoutservice_delay_1""","""RE2-OB""","""RE2""","""Online Boutique""","""delay""","""network delay""","""checkoutservice""",1705666511,1705665791,1705667231,24.0,720,721
"""re2ob_checkoutservice_loss_1""","""RE2-OB""","""RE2""","""Online Boutique""","""loss""","""network packet loss""","""checkoutservice""",1705376266,1705375546,1705376986,24.0,720,721
"""re2ob_checkoutservice_socket_1""","""RE2-OB""","""RE2""","""Online Boutique""","""socket""","""socket exhaustion""","""checkoutservice""",1705656313,1705655593,1705657033,24.0,720,721
"""re2ob_currencyservice_cpu_1""","""RE2-OB""","""RE2""","""Online Boutique""","""cpu""","""CPU stress""","""currencyservice""",1705682817,1705682097,1705683537,24.0,720,721


In [6]:
print("Répartition des faults:")
display(
    ground_truth
    .group_by("fault")
    .len()
    .sort("fault")
)

Répartition des faults:


fault,len
str,u32
"""cpu""",15
"""delay""",15
"""loss""",15
"""socket""",15


## 6. Test du DatasetLoader

On vérifie qu'un incident complet peut être chargé avec ses métriques, logs et traces.


In [7]:
TEST_CASE = "re2ob_checkoutservice_cpu_1"

info = loader.get_case_info(TEST_CASE)

print(info)

CaseInfo(case_id='re2ob_checkoutservice_cpu_1', dataset='RE2-OB', fault='cpu', root_cause_service='checkoutservice', time_start_ms=1705353846000, inject_time_ms=1705354566000, time_end_ms=1705355286000, incident_type=None)


In [8]:
data = loader.load_case(
    TEST_CASE,
    include_metrics=True,
    include_logs=True,
    include_traces=True,
    metrics_long_format=True,
)

print("Metrics:", data["metrics"].shape)
print("Logs:", data["logs"].shape)
print("Traces:", data["traces"].shape)

Metrics: (785345, 8)
Logs: (171322, 8)
Traces: (391997, 16)


## 7. Vérification de la fenêtre temporelle


In [9]:
metrics = data["metrics"]
logs = data["logs"]
traces = data["traces"]

print("Ground truth")
print("Start :", info.time_start_ms)
print("Inject:", info.inject_time_ms)
print("End   :", info.time_end_ms)

print("\nMetrics")
print("Min:", metrics["timestamp"].min())
print("Max:", metrics["timestamp"].max())

print("\nLogs")
print("Min:", logs["timestamp_ms"].min())
print("Max:", logs["timestamp_ms"].max())

print("\nTraces")
print("Min:", traces["timestamp_ms"].min())
print("Max:", traces["timestamp_ms"].max())

Ground truth
Start : 1705353846000
Inject: 1705354566000
End   : 1705355286000

Metrics
Min: 1705353846000
Max: 1705355286000

Logs
Min: 1705353846000
Max: 1705355286000

Traces
Min: 1705353846065
Max: 1705355285970


## 8. Normalisation des métriques

Le normalizer transforme les métriques en événements canoniques ECDT.


In [10]:
normalized_metrics = normalizer.normalize_metrics(metrics)

print("Shape:", normalized_metrics.shape)
print("Schema:")
print(normalized_metrics.schema)

Shape: (785345, 19)
Schema:
Schema({'event_id': String, 'case_id': String, 'timestamp_ms': Int64, 'dataset': String, 'fault': String, 'root_cause_service': String, 'source': String, 'service_name': String, 'signal_type': String, 'metric_name': String, 'value': Float64, 'message': String, 'trace_id': String, 'span_id': String, 'parent_span_id': String, 'method_name': String, 'operation_name': String, 'duration_ms': Float64, 'status_code': Float64})


In [11]:
display(
    normalized_metrics.select([
        "source",
        "service_name",
        "signal_type",
        "metric_name",
        "value",
        "timestamp_ms",
    ]).head(10)
)

source,service_name,signal_type,metric_name,value,timestamp_ms
str,str,str,str,f64,i64
"""metric""","""adservice""","""cpu""","""adservice_cpu""",0.685023,1705353846000
"""metric""","""adservice""","""cpu""","""adservice_cpu""",0.685023,1705353847000
"""metric""","""adservice""","""cpu""","""adservice_cpu""",0.582707,1705353848000
"""metric""","""adservice""","""cpu""","""adservice_cpu""",0.625903,1705353849000
"""metric""","""adservice""","""cpu""","""adservice_cpu""",0.613414,1705353850000
"""metric""","""adservice""","""cpu""","""adservice_cpu""",0.600926,1705353851000
"""metric""","""adservice""","""cpu""","""adservice_cpu""",0.588438,1705353852000
"""metric""","""adservice""","""cpu""","""adservice_cpu""",0.57595,1705353853000
"""metric""","""adservice""","""cpu""","""adservice_cpu""",0.563462,1705353854000


## 9. Vérification du parsing des métriques


In [12]:
metric_examples = [
    "adservice_cpu",
    "checkoutservice_mem",
    "carts-db_diskio",
    "frontend_latency-50",
    "ts-route-service_latency-90",
    "ts-auth-service_error",
]

for name in metric_examples:
    print(f"{name:40} -> {normalizer.parse_metric_name(name)}")

adservice_cpu                            -> ('adservice', 'cpu')
checkoutservice_mem                      -> ('checkoutservice', 'mem')
carts-db_diskio                          -> ('carts-db', 'diskio')
frontend_latency-50                      -> ('frontend', 'latency-50')
ts-route-service_latency-90              -> ('ts-route-service', 'latency-90')
ts-auth-service_error                    -> ('ts-auth-service', 'error')


## 10. Normalisation complète

On vérifie que métriques, logs et traces peuvent être fusionnés dans le schéma canonique ECDT.


In [13]:
events_df = normalizer.normalize_all(
    metrics=data["metrics"],
    logs=data["logs"],
    traces=data["traces"],
)

print("Normalized events:", events_df.shape)
print("\nSources:")
display(events_df.group_by("source").len().sort("source"))

Normalized events: (1348664, 19)

Sources:


source,len
str,u32
"""log""",171322
"""metric""",785345
"""trace""",391997


## 11. Vérification du schéma canonique


In [14]:
expected_columns = {
    "event_id",
    "case_id",
    "timestamp_ms",
    "dataset",
    "fault",
    "root_cause_service",
    "source",
    "service_name",
    "signal_type",
    "metric_name",
    "value",
    "message",
    "trace_id",
    "span_id",
    "parent_span_id",
    "method_name",
    "operation_name",
    "duration_ms",
    "status_code",
}

actual_columns = set(events_df.columns)

print("Missing columns:", expected_columns - actual_columns)
print("Unexpected columns:", actual_columns - expected_columns)

assert expected_columns.issubset(actual_columns)
print("SCHEMA CHECK: PASS")

Missing columns: set()
Unexpected columns: set()
SCHEMA CHECK: PASS


## 12. Construction des séries temporelles

Le detector construit une série par couple `(service, signal_type)` à partir des événements métriques.


In [15]:
events = events_df.to_dicts()
series = detector.build_series(events)

print("Number of time series:", len(series))
print("\nFirst series:")
for key in list(series.keys())[:20]:
    print(" -", key)

Number of time series: 72

First series:
 - ('recommendationservice', 'latency-90')
 - ('currencyservice', 'cpu')
 - ('productcatalogservice', 'latency-90')
 - ('paymentservice', 'latency-90')
 - ('frontend', 'latency-90')
 - ('emailservice', 'cpu')
 - ('emailservice', 'latency-90')
 - ('currencyservice', 'latency-90')
 - ('checkoutservice', 'latency-90')
 - ('cartservice', 'latency-90')
 - ('frontend', 'cpu')
 - ('adservice', 'latency-90')
 - ('shippingservice', 'latency-50')
 - ('recommendationservice', 'latency-50')
 - ('productcatalogservice', 'latency-50')
 - ('paymentservice', 'cpu')
 - ('paymentservice', 'latency-50')
 - ('frontend', 'latency-50')
 - ('emailservice', 'latency-50')
 - ('productcatalogservice', 'cpu')


## 13. Validation de la série CPU du service root cause


In [16]:
key = (info.root_cause_service, "cpu")

assert key in series, f"Series not found: {key}"

ts = series[key]

print("Series:", key)
print("Points:", len(ts))
print("Start:", min(ts.timestamps))
print("End:", max(ts.timestamps))
print("Injection:", info.inject_time_ms)

Series: ('checkoutservice', 'cpu')
Points: 1441
Start: 1705353846000.0
End: 1705355286000.0
Injection: 1705354566000


## 14. Construction du baseline

Règle importante : le baseline doit être construit exclusivement avec les observations précédant l'injection.


In [17]:
baseline_values = [
    value
    for timestamp, value
    in zip(ts.timestamps, ts.values)
    if timestamp < info.inject_time_ms
]

baseline_mean, baseline_std, baseline_max = detector.compute_baseline(
    baseline_values
)

print("Baseline samples:", len(baseline_values))
print("Mean:", baseline_mean)
print("Std:", baseline_std)
print("Max:", baseline_max)

assert len(baseline_values) > 0
print("BASELINE CHECK: PASS")

Baseline samples: 720
Mean: 0.42885440894546123
Std: 0.07882683449695392
Max: 0.6869909369114802
BASELINE CHECK: PASS


## 15. Validation du Z-score


In [18]:
print("z(10 | 10, 1) =", detector.zscore(10.0, 10.0, 1.0))
print("z(11 | 10, 1) =", detector.zscore(11.0, 10.0, 1.0))
print("z(13 | 10, 1) =", detector.zscore(13.0, 10.0, 1.0))
print("z(7  | 10, 1) =", detector.zscore(7.0, 10.0, 1.0))

assert detector.zscore(10.0, 10.0, 1.0) == 0.0
assert detector.zscore(13.0, 10.0, 1.0) == 3.0
assert detector.zscore(7.0, 10.0, 1.0) == -3.0

print("ZSCORE CHECK: PASS")

z(10 | 10, 1) = 0.0
z(11 | 10, 1) = 1.0
z(13 | 10, 1) = 3.0
z(7  | 10, 1) = -3.0
ZSCORE CHECK: PASS


## 16. Détection des anomalies


In [19]:
anomalies = detector.detect_series(
    ts,
    baseline_end=info.inject_time_ms,
)

before = [
    a for a in anomalies
    if a.timestamp < info.inject_time_ms
]

after = [
    a for a in anomalies
    if a.timestamp >= info.inject_time_ms
]

print("Total anomalies:", len(anomalies))
print("Before injection:", len(before))
print("After injection:", len(after))

Total anomalies: 707
Before injection: 0
After injection: 707


## 17. Validation du CPU incident

Le cas de référence doit produire des anomalies après l'injection et aucune anomalie avant l'injection.


In [20]:
assert len(after) > 0, "No anomaly detected after injection"
assert len(before) == 0, "False positive detected before injection"

first = min(after, key=lambda a: a.timestamp)
last = max(after, key=lambda a: a.timestamp)

detection_delay = (
    first.timestamp - info.inject_time_ms
) / 1000

print("First detection:", first.timestamp)
print("Detection delay:", detection_delay, "seconds")
print("Last detection:", last.timestamp)
print("First anomaly value:", first.value)
print("First anomaly score:", first.score)
print("Incident type:", first.incident_type)

print("CPU DETECTION CHECK: PASS")

First detection: 1705354580000.0
Detection delay: 14.0 seconds
Last detection: 1705355286000.0
First anomaly value: 5.632195848767665
First anomaly score: 66.00977285245769
Incident type: IncidentType.CPU_SATURATION
CPU DETECTION CHECK: PASS


## 18. Validation des quatre scénarios ECDT

Cette section reproduit la validation réalisée dans `tests/test_incidents.py`.

Pour conserver la cohérence avec le test déjà validé, la série `checkoutservice/cpu` est utilisée ici comme signal d'observation commun. Cette validation démontre la capacité du détecteur à identifier une rupture statistique autour de l'injection ; elle ne constitue pas encore une évaluation complète du mapping télémétrique spécifique à chaque fault.


In [21]:
TEST_CASES = {
    "cpu": "re2ob_checkoutservice_cpu_1",
    "delay": "re2ob_checkoutservice_delay_1",
    "loss": "re2ob_checkoutservice_loss_1",
    "socket": "re2ob_checkoutservice_socket_1",
}

results = []

for fault, case in TEST_CASES.items():
    info = loader.get_case_info(case)

    case_data = loader.load_case(
        case,
        include_metrics=True,
        include_logs=False,
        include_traces=False,
        metrics_long_format=True,
    )

    normalized = normalizer.normalize_all(
        metrics=case_data["metrics"]
    )

    case_series = detector.build_series(normalized.to_dicts())
    ts = case_series[(info.root_cause_service, "cpu")]

    baseline_values = [
        value
        for timestamp, value
        in zip(ts.timestamps, ts.values)
        if timestamp < info.inject_time_ms
    ]

    baseline_mean, baseline_std, baseline_max = detector.compute_baseline(
        baseline_values
    )

    anomalies = detector.detect_series(
        ts,
        baseline_end=info.inject_time_ms,
    )

    before = [
        a for a in anomalies
        if a.timestamp < info.inject_time_ms
    ]

    after = [
        a for a in anomalies
        if a.timestamp >= info.inject_time_ms
    ]

    delay = None
    if after:
        first = min(after, key=lambda a: a.timestamp)
        delay = (first.timestamp - info.inject_time_ms) / 1000

    results.append({
        "fault": fault,
        "case": case,
        "baseline_mean": baseline_mean,
        "baseline_std": baseline_std,
        "anomalies": len(anomalies),
        "before": len(before),
        "after": len(after),
        "detection_delay_s": delay,
    })

results_df = pl.DataFrame(results)
display(results_df)

fault,case,baseline_mean,baseline_std,anomalies,before,after,detection_delay_s
str,str,f64,f64,i64,i64,i64,f64
"""cpu""","""re2ob_checkoutservice_cpu_1""",0.428854,0.078827,707,0,707,14.0
"""delay""","""re2ob_checkoutservice_delay_1""",0.422945,0.075131,15,0,15,188.0
"""loss""","""re2ob_checkoutservice_loss_1""",0.428367,0.0848,51,0,51,58.0
"""socket""","""re2ob_checkoutservice_socket_1""",0.393426,0.075372,236,0,236,7.0


## 19. Critères de validation


In [22]:
assert results_df.height == 4
assert results_df["after"].min() > 0
assert results_df["before"].max() == 0

print("All four scenarios produced post-injection detections.")
print("No pre-injection false positives were observed.")
print("FINAL DETECTOR CHECK: PASS")

All four scenarios produced post-injection detections.
No pre-injection false positives were observed.
FINAL DETECTOR CHECK: PASS


## 20. Résumé de validation

Les critères suivants sont considérés comme validés :

- DatasetLoader opérationnel ;
- chargement des métriques ;
- chargement des logs ;
- chargement des traces ;
- conversion temporelle cohérente ;
- normalisation des métriques ;
- parsing service / signal ;
- fusion dans un schéma canonique ;
- construction des séries temporelles ;
- calcul du baseline ;
- calcul du Z-score ;
- détection d'anomalies ;
- aucune anomalie avant injection dans les quatre tests ;
- détection après injection dans les quatre tests.

La détection d'anomalies constitue cependant une étape de prétraitement. Elle ne constitue pas encore le moteur RCA complet du Digital Twin.

## 21. Résultats de référence obtenus pendant la validation

Pour le cas CPU `re2ob_checkoutservice_cpu_1`, les résultats de référence sont :

```text
Series points       : 1441
Baseline samples    : 720
Baseline mean       : 0.428854
Baseline std        : 0.078827
Total anomalies     : 707
Before injection    : 0
After injection     : 707
Detection delay     : 14 s
First anomaly value : 5.632196
First anomaly score : 66.009773
```

Pour les quatre scénarios testés sur `checkoutservice/cpu` :

| Fault | Anomalies | Before | After | Delay |
|---|---:|---:|---:|---:|
| CPU | 707 | 0 | 707 | 14 s |
| DELAY | 15 | 0 | 15 | 188 s |
| LOSS | 51 | 0 | 51 | 58 s |
| SOCKET | 236 | 0 | 236 | 7 s |
        

## 22. Conclusion de la Phase 2 — étape ingestion

La chaîne d'ingestion et de détection statistique est fonctionnelle sur le sous-ensemble ECDT.

```text
DatasetLoader              PASS
SchemaNormalizer           PASS
Metric parsing             PASS
Event normalization        PASS
Time-series construction   PASS
Baseline computation       PASS
Z-score                    PASS
Pre-injection filtering    PASS
Post-injection detection   PASS
Four incident tests        PASS

Le détecteur peut maintenant être considéré comme une première couche opérationnelle de détection d'anomalies pour ECDT.

Les améliorations de performance, notamment la réduction du délai de détection du scénario DELAY, pourront être traitées dans une étape ultérieure et ne bloquent pas la clôture de cette étape.

La prochaine étape consiste à exploiter les événements normalisés et les anomalies détectées pour construire la représentation du Digital Twin, puis les mécanismes de corrélation et de Root Cause Analysis.